Idea:

- Turn each meal into Darts' series with time as covariate
- Each restaurant corresponds with a dedicated forecasting model

In [ ]:
%cd ../../

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import joblib
from statsmodels.graphics.tsaplots import plot_acf
import polars as pl
from darts.models import NaiveMovingAverage, LinearRegressionModel, RandomForest, KalmanForecaster
from darts import TimeSeries
from darts.utils.utils import generate_index
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import MinMaxScaler
from darts.dataprocessing import Pipeline
from darts.metrics import mape
from tqdm import tqdm

In [ ]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})


In [ ]:
CUTOFF_DATE = "2025-01-01"

# Load dims and raw data

## dim `meals`

In [ ]:
dim_meals = pl.read_parquet("data/processed/dim_meals.parquet")
dim_meals.head()

## fact `pos`

In [ ]:
path = "data/processed/pos.xlsx"
pos_raw = pl.read_excel(path)

pos_raw.head()

In [ ]:
pos = (
    pos_raw
    .with_columns(
        pl.col('datetime').dt.date().alias('date')
    )
    .group_by('restaurant', 'date', 'meal_id')
    .agg(pl.col('pcs').sum())
)


# Only keep entries of meals having data since CUTOFF_DATE
meals_concerned = (
    pos
    .filter(pl.col('date') >= pl.lit(CUTOFF_DATE, dtype=pl.Date))
    .select('meal_id', 'restaurant').unique()
)

pos = pos.join(meals_concerned, on=['meal_id', 'restaurant'], how='inner')


# Remove pos of leftover meals
pos = (
    pos
    .with_columns(
        pl.col('date').shift(1).over('restaurant', 'meal_id', order_by='date').alias('date_lag'),
        pl.col('pcs').shift(1).over('restaurant', 'meal_id', order_by='date').alias('pcs_lag'),
    )
    .filter(
        (pl.col('date_lag').is_null())
        | ((pl.col('date') - pl.col('date_lag')).dt.total_days() > 7)
    )
    .drop('date_lag', 'pcs_lag')
)


# Keep records whose POS value is greater than Q1 value
pos = (
    pos
    .with_columns(
        pl.col('pcs').quantile(.25).over('restaurant', 'meal_id', order_by='date').alias('pcs_q1'),
        pl.col('pcs').quantile(.75).over('restaurant', 'meal_id', order_by='date').alias('pcs_q3')
    )
    .filter(
        (1 == 1)
        & (pl.col('pcs') >= pl.col('pcs_q1'))
        & (pl.col('pcs') <= pl.col('pcs_q3'))
    )
    .drop('pcs_q1', 'pcs_q3')
)



# Add datetime attributes
pos = (
    pos
    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
        pl.col('date').dt.day().alias('day'),
        pl.col('date').dt.week().alias('week'),
        pl.col('date').dt.month().alias('month'),
        pl.col('date').dt.year().alias('year'),
    )
    .with_columns(
        (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
        (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
        (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
        (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
        (pl.col('week') * 2 * np.pi / 53).sin().alias('week_sin'),
        (pl.col('week') * 2 * np.pi / 53).cos().alias('week_cos'),
        (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
        (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
    )
)


pos.head()

# Build Time series

In [ ]:
col_tgt = 'pcs'

cols_cov = [
    'weekday',
    'day',
    'week',
    'month',
    'year',

    'weekday_sin',
    'weekday_cos',
    'day_sin',
    'day_cos',
    'week_sin',
    'week_cos',
    'month_sin',
    'month_cos',
]

In [ ]:
model_name = "rf"
THRES = 3

results = []
pipeline = Pipeline([
    # InvertibleMapper(np.log1p, np.expm1, verbose=False, n_jobs=-1, name="Log-Transform"),
    Scaler(verbose=False, n_jobs=-1, name="Scaling"),
])

# pairs = pos.select('restaurant', 'meal_id').unique()
for pair in tqdm(meals_concerned.iter_rows(named=True), total=len(meals_concerned)):
    df = (
        pos
        .filter(
            (1 == 1)
            & (pl.col('meal_id') == pair['meal_id'])
            & (pl.col('restaurant') == pair['restaurant'])
            & (pl.col(col_tgt) > 0)
        )
        .sort('date')
        .with_row_index('index')
        .to_pandas()
    )

    if len(df) < 3:
        continue

    series = TimeSeries.from_dataframe(
        df,
        time_col='index',
        value_cols=col_tgt,
        fillna_value=False,
        # static_covariates=pd.DataFrame(data={"meal_type": [entry['meal_type']]})
    )

    series_cov = TimeSeries.from_dataframe(
        df,
        time_col='index',
        value_cols=cols_cov,
        fillna_value=False,
        # static_covariates=pd.DataFrame(data={"meal_type": [entry['meal_type']]})
    )

    cutpoint = len(df[df['date'] <= CUTOFF_DATE])
    if cutpoint <= THRES or cutpoint == len(df) :
        continue


    series_train, series_test = series.split_before(cutpoint)


    series_train_transformed = pipeline.fit_transform(series_train)

    lags = min(5, len(series_train) - 1)
    match model_name:
        case "naive-moving":
            model = NaiveMovingAverage(lags)
            model.fit(series_train_transformed)
            preds = model.predict(len(series_test))
        case "linear":
            model = LinearRegressionModel(lags, lags_future_covariates=[0])
            model.fit(series_train_transformed, future_covariates=series_cov)
            preds = model.predict(len(series_test), future_covariates=series_cov)
        case "rf":
            model = RandomForest(lags, lags_future_covariates=[0])
            model.fit(series_train_transformed, future_covariates=series_cov)
            preds = model.predict(len(series_test), future_covariates=series_cov)
        case "kalman":
            model = KalmanForecaster(lags)
            model.fit(series_train_transformed, future_covariates=series_cov)
            preds = model.predict(len(series_test), future_covariates=series_cov)

    preds = pipeline.inverse_transform(preds)

    # Eval
    mape_val = mape(series_test, preds)

    # fig = plt.figure(figsize=(8, 6))
    # ax = fig.add_subplot(111)

    # series_train.plot(ax=ax, label='train')
    # series_test.plot(ax=ax, label='test-gt')
    # preds.plot(ax=ax, label='test-pred')

    # fig.suptitle(f"meal_id = {meal_id['meal_id']} | MAPE = {mape_val.item():.4f}", fontsize=20, fontweight='bold')
    

    results.append({**pair, 'mape': mape_val.item()})

In [ ]:
(
    pl
    .from_dicts(results)
    # .sort('mape', descending=True)
    .group_by('restaurant')
    .agg(pl.col('mape').mean())
)
    # .head()

In [ ]:
meal_id = 10
restaurant = 1

df = (
    pos
    .filter(
        (1 == 1)
        & (pl.col('meal_id') == meal_id)
        & (pl.col('restaurant') == restaurant)
        & (pl.col(col_tgt) > 0)
    )
    .sort('date')
    .to_pandas()
)

series = TimeSeries.from_dataframe(
    df,
    value_cols=col_tgt,
    fillna_value=False,
    # static_covariates=pd.DataFrame(data={"meal_type": [entry['meal_type']]})
)

series_cov = TimeSeries.from_dataframe(
    df,
    value_cols=cols_cov,
    fillna_value=False,
    # static_covariates=pd.DataFrame(data={"meal_type": [entry['meal_type']]})
)

cutpoint = len(df[df['date'] <= CUTOFF_DATE])


series_train, series_test = series.split_before(cutpoint)

pipeline = Pipeline([
    # InvertibleMapper(np.log1p, np.expm1, verbose=False, n_jobs=-1, name="Log-Transform"),
    Scaler(verbose=False, n_jobs=-1, name="Scaling"),
])
series_train_transformed = pipeline.fit_transform(series_train)

lags = min(5, len(series_train))
match model_name:
    case "naive-moving":
        model = NaiveMovingAverage(lags)
        model.fit(series_train_transformed)
        preds = model.predict(len(series_test))
    case "linear":
        model = LinearRegressionModel(lags, lags_future_covariates=[0])
        model.fit(series_train_transformed, future_covariates=series_cov)
        preds = model.predict(len(series_test), future_covariates=series_cov)
    case "rf":
        model = RandomForest(lags, lags_future_covariates=[0])
        model.fit(series_train_transformed, future_covariates=series_cov)
        preds = model.predict(len(series_test), future_covariates=series_cov)
    case "kalman":
        model = KalmanForecaster(lags)
        model.fit(series_train_transformed, future_covariates=series_cov)
        preds = model.predict(len(series_test), future_covariates=series_cov)

preds = pipeline.inverse_transform(preds)

# Eval
mape_val = mape(series_test, preds)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

series_train.plot(ax=ax, label='train')
series_test.plot(ax=ax, label='test-gt')
preds.plot(ax=ax, label='test-pred')

fig.suptitle(f"meal_id = {meal_id} | MAPE = {mape_val.item():.4f}", fontsize=20, fontweight='bold')

# Export to production model

In [ ]:
tag = "May_26"

path_dir = Path("trained_models/per_meal_pos") / tag

## Models for meals having historical data

In [ ]:
col_tgt = 'pcs'

In [ ]:
THRES = 4
LAG = 3

for pair in tqdm(pos.select('restaurant', 'meal_id').unique().iter_rows(named=True), total=len(meals_concerned)):
    df = (
        pos
        .filter(
            (1 == 1)
            & (pl.col('meal_id') == pair['meal_id'])
            & (pl.col('restaurant') == pair['restaurant'])
            & (pl.col(col_tgt) > 0)
        )
        .sort('date')
        .with_row_index(offset=1)
        .to_pandas()
    )
    if len(df) == 0:
        continue


    # Create and transform series of target and covariate
    series = TimeSeries.from_times_and_values(
        times=generate_index(0, length=len(df)),
        values=df[col_tgt].to_numpy(),
        columns=col_tgt
    )


    # Choose NaiveMovingAverage
    model = NaiveMovingAverage(input_chunk_length=min(len(df), LAG))
    model.fit(series)


    # Save models
    path_dir_pair = path_dir / f"{pair['restaurant']}_{pair['meal_id']}"
    path_dir_pair.mkdir(exist_ok=True, parents=True)

    path_model = path_dir_pair / "model.pt"
    model.save(path_model.as_posix(), clean=False)


### Test loading model and making prediction

In [ ]:
date = '2025-06-05'
restaurant = 1
meal_id = 51


# Load saved model
path_dir_pair = path_dir / f"{restaurant}_{meal_id}"

path_model = path_dir_pair / "model.pt"

model = NaiveMovingAverage.load(path_model)    
preds = model.predict(1)

preds


## Models having no historical data

***[May 28]***
- Use median of all meals belonging to same restaurant and having same meal_type


In [ ]:
path_general_models = path_dir / "general_pos.csv"

(
    pos
    .join(
        dim_meals.select(pl.col('id').alias('meal_id'), 'meal_type'),
        on='meal_id',
        how='left'
    )
    .group_by('restaurant', 'meal_type')
    .agg(pl.col(col_tgt).median())
    .write_csv(path_general_models)
)